# 오늘의집 집들이 EDA
## 분석 목적
오늘의집 집들이 게시글(8,000건)과 상품 태그 데이터(194만건)를 활용하여  
**사용자 맞춤 인테리어 상품 조합 추천이 가능한지** 검증한다.

### 검증 질문
1. 사용자 조건(거주유형·스타일·색조·예산)으로 필터링했을 때 충분한 게시글이 남는가?
2. 예산 필터링 기준(카테고리별 단가 / 공간별 합산)이 현실적으로 작동하는가?
3. 색조·스타일별로 등장하는 상품 패턴이 실제로 다른가? (조합 추천의 근거)
4. 공간별로 자주 함께 등장하는 상품 카테고리 조합이 존재하는가?
5. 인기도 기반 상품 랭킹이 의미있게 작동하는가?

### 데이터
- `housewarming_posts_with_tone_urlinfo.csv` — 게시글 8,000건
- `housewarming_product_tags_with_image.csv` — 상품 태그 194만건
- `housewarming_posts_final.csv` — EDA 과정에서 생성된 최종 게시글 데이터
- `housewarming_product_tags_clean.csv` — EDA 과정에서 생성된 클린 태그 데이터


## 공통 설정
세팅, 데이터 로드, 전처리 함수를 정의한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import ast
import warnings
from itertools import combinations
from collections import Counter
warnings.filterwarnings("ignore")

# ── 한글 폰트 ──
def set_korean_font():
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Gulim"]
    available = {f.name for f in fm.fontManager.ttflist}
    for font in candidates:
        if font in available:
            plt.rcParams["font.family"] = font
            break
    plt.rcParams["axes.unicode_minus"] = False

set_korean_font()

BASE = r"C:\Users\SAMSUNG\Desktop\ohouse_final"

# ── 공통 함수 ──
def parse_list_col(series):
    def _parse(x):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return series.apply(_parse)

def group_tone(tone):
    if pd.isna(tone): return None
    tone = str(tone)
    if "Gray" in tone or "White" in tone or "Black" in tone: return "무채색"
    elif "Orange" in tone or "Yellow" in tone: return "웜톤"
    elif "Blue" in tone or "Sky" in tone: return "쿨톤"
    elif "Green" in tone: return "그린톤"
    else: return "기타"

def save_fig(name, fig):
    path = f"{BASE}\\{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"저장: {name}.png")

print("설정 완료")

게시글: 8,000행 × 26열
전처리 완료


---
## EDA 01. 필터 조건 유효성 검증
**목적**: 사용자가 입력하는 조건(거주유형·스타일·색조·예산·지역)으로 필터링했을 때  
추천에 쓸 수 있는 게시글이 충분히 남는지 확인한다.

**결론**:
- 거주유형·색조·전문가여부·공사유형은 커버리지 100%로 핵심 필터로 사용 가능
- 예산(features_budget)은 입력률 42.3%로 낮아 선택 필터로만 사용
- 지역은 수도권 편중이 심해 soft 필터(가중치)로만 사용
- 조건 3개(거주+스타일+색조)까지는 충분, 4개부터 조심


In [ ]:
# ── 데이터 로드 ──
posts = pd.read_csv(f"{BASE}\\housewarming_posts_with_tone_urlinfo.csv")
print(f"게시글: {posts.shape[0]:,}행 × {posts.shape[1]}열")

# ── 전처리 ──
posts["styleList"]    = parse_list_col(posts["features_styleList"])
posts["area_num"]     = posts["features_area"].str.extract(r"(\d+)").astype(float)
posts["budget_clean"] = posts["features_budget"].replace(0, np.nan)
posts["style_main"]   = posts["styleList"].apply(lambda x: x[0] if len(x) > 0 else None)
posts["tone_group"]   = posts["primary_tone"].apply(group_tone)
posts["sido"]         = posts["features_region"].str.extract(r"^([\w]+(?:특별시|광역시|도|특별자치도|특별자치시)?)")

budget_bins   = [0, 500, 1000, 2000, 3000, 5000, 10000, np.inf]
budget_labels = ["~500만", "500~1000만", "1000~2000만", "2000~3000만",
                 "3000~5000만", "5000만~1억", "1억+"]
posts["budget_range"] = pd.cut(posts["budget_clean"], bins=budget_bins, labels=budget_labels)
n_total = len(posts)
print("전처리 완료")

게시글: 8,000행 × 26열
전처리 완료


In [ ]:
# ── [1-1] 예산 입력률 ──
n_valid   = posts["budget_clean"].notna().sum()
n_missing = n_total - n_valid
print(f"[1-1] 예산 입력률")
print(f"  입력: {n_valid:,}건 ({n_valid/n_total*100:.1f}%)")
print(f"  미입력: {n_missing:,}건 ({n_missing/n_total*100:.1f}%)")

budget_dist = posts["budget_range"].value_counts().sort_index()
for label, cnt in budget_dist.items():
    print(f"    {str(label):>15s}: {cnt:>5,}건 ({cnt/n_valid*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("[1-1] 예산 입력률 및 구간 분포", fontsize=14, fontweight="bold")
axes[0].pie([n_valid, n_missing],
            labels=[f"입력\n{n_valid:,}건", f"미입력\n{n_missing:,}건"],
            autopct="%1.1f%%", colors=["#4C72B0", "#D0D0D0"],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("예산 입력 여부")
axes[1].bar(range(len(budget_dist)), budget_dist.values, color="#4C72B0", edgecolor="white")
axes[1].set_xticks(range(len(budget_dist)))
axes[1].set_xticklabels(budget_dist.index.astype(str), rotation=35, ha="right")
axes[1].set_title("예산 구간별 게시글 수 (입력 게시글만)")
axes[1].set_ylabel("게시글 수")
for i, v in enumerate(budget_dist.values):
    axes[1].text(i, v + 5, f"{v:,}", ha="center", fontsize=9)
save_fig("eda_01_1_budget", fig)

# ── [1-2] 단일 필터 커버리지 ──
print("\n[1-2] 단일 필터 조건별 커버리지")
single_filters = {
    "거주유형": "features_residence", "대표스타일": "style_main",
    "주색조": "primary_tone", "전문가여부": "features_agent",
    "공사유형": "features_expertise", "예산구간": "budget_range", "지역": "features_region",
}
print(f"  {'조건':<12} {'유니크값':>6} {'유효 게시글':>10} {'커버리지':>8}")
print("  " + "-" * 42)
for label, col in single_filters.items():
    n_col  = posts[col].notna().sum()
    n_uniq = posts[col].nunique()
    print(f"  {label:<12} {n_uniq:>6,} {n_col:>10,} {n_col/n_total*100:>7.1f}%")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("[1-2] 단일 필터 조건별 분포", fontsize=14, fontweight="bold")
axes = axes.flatten()
plot_cols = [("거주유형","features_residence"),("대표스타일","style_main"),
             ("주색조","primary_tone"),("전문가여부","features_agent"),
             ("공사유형","features_expertise"),("예산구간","budget_range")]
for i, (label, col) in enumerate(plot_cols):
    vc = posts[col].value_counts().head(12)
    axes[i].barh(vc.index.astype(str)[::-1], vc.values[::-1], color="#4C72B0", edgecolor="white")
    axes[i].set_title(label)
    axes[i].set_xlabel("게시글 수")
    for j, v in enumerate(vc.values[::-1]):
        axes[i].text(v + 5, j, f"{v:,}", va="center", fontsize=8)
save_fig("eda_01_2_single_filter", fig)

# ── [1-3] 조건 조합 시나리오 ──
print("\n[1-3] 조건 조합 필터링 시 잔존 게시글 수")
scenarios = [
    {"name": "거주유형만 (아파트)",          "filters": {"features_residence": "아파트"}},
    {"name": "아파트 + 내추럴",             "filters": {"features_residence": "아파트", "style_main": "내추럴"}},
    {"name": "아파트 + 내추럴 + Light-Gray", "filters": {"features_residence": "아파트", "style_main": "내추럴", "primary_tone": "Light-Gray"}},
    {"name": "아파트 + 내추럴 + Light-Gray + 예산", "filters": {"features_residence": "아파트", "style_main": "내추럴", "primary_tone": "Light-Gray", "budget_range": "1000~2000만"}},
    {"name": "아파트 + 내추럴 + Light-Gray + 예산 + 서울", "filters": {"features_residence": "아파트", "style_main": "내추럴", "primary_tone": "Light-Gray", "budget_range": "1000~2000만", "features_region": "서울"}},
    {"name": "원룸 + 모던",                 "filters": {"features_residence": "원룸&오피스텔", "style_main": "모던"}},
    {"name": "원룸 + 모던 + Medium-Gray",   "filters": {"features_residence": "원룸&오피스텔", "style_main": "모던", "primary_tone": "Medium-Gray"}},
    {"name": "원룸 + 모던 + Medium-Gray + ~500만", "filters": {"features_residence": "원룸&오피스텔", "style_main": "모던", "primary_tone": "Medium-Gray", "budget_range": "~500만"}},
]
results = []
print(f"  {'시나리오':<42} {'잔존':>6} {'비율':>6}  판정")
print("  " + "-" * 65)
for s in scenarios:
    sub = posts.copy()
    for col, val in s["filters"].items():
        if col == "features_region":
            sub = sub[sub[col].fillna("").str.contains(val, na=False)]
        else:
            sub = sub[sub[col] == val]
    n = len(sub)
    flag = "✅ 충분" if n >= 10 else "⚠️ 부족" if n >= 3 else "❌ 불가"
    print(f"  {s['name']:<42} {n:>6,} {n/n_total*100:>5.2f}%  {flag}")
    results.append({"시나리오": s["name"], "잔존": n, "비율": round(n/n_total*100,2), "판정": flag})
df_results = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle("[1-3] 조건 조합별 잔존 게시글 수", fontsize=14, fontweight="bold")
bar_colors = ["#4C72B0" if r >= 10 else "#DD8452" if r >= 3 else "#D94F4F" for r in df_results["잔존"]]
bars = ax.barh(df_results["시나리오"][::-1], df_results["잔존"][::-1], color=bar_colors[::-1], edgecolor="white")
ax.axvline(x=10, color="green", linestyle="--", linewidth=1.2, label="최소 기준 (10건)")
ax.axvline(x=3,  color="orange", linestyle="--", linewidth=1.2, label="위험 기준 (3건)")
ax.set_xlabel("잔존 게시글 수")
ax.legend()
for bar, v in zip(bars, df_results["잔존"][::-1]):
    ax.text(v + 2, bar.get_y() + bar.get_height()/2, f"{v:,}건", va="center", fontsize=9)
save_fig("eda_01_3_combo", fig)

# ── [1-4] 지역 커버리지 ──
print("\n[1-4] 지역 커버리지")
region_valid = posts["features_region"].notna().sum()
region_missing = n_total - region_valid
print(f"  지역 입력: {region_valid:,}건 ({region_valid/n_total*100:.1f}%)")
print(f"  미입력: {region_missing:,}건 ({region_missing/n_total*100:.1f}%)")
sido_counts = posts["sido"].value_counts().head(15)
print(f"\n  상위 10개 시/도:\n{sido_counts.head(10).to_string()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("[1-4] 지역 커버리지", fontsize=14, fontweight="bold")
axes[0].pie([region_valid, region_missing],
            labels=[f"입력\n{region_valid:,}건", f"미입력\n{region_missing:,}건"],
            autopct="%1.1f%%", colors=["#4C72B0", "#D0D0D0"],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("지역 입력 여부")
axes[1].barh(sido_counts.index[::-1], sido_counts.values[::-1], color="#4C72B0", edgecolor="white")
axes[1].set_title("시/도별 게시글 수 (상위 15)")
axes[1].set_xlabel("게시글 수")
for i, v in enumerate(sido_counts.values[::-1]):
    axes[1].text(v + 5, i, f"{v:,}", va="center", fontsize=9)
save_fig("eda_01_4_region", fig)
print("\nEDA 01 완료!")


[1-1] 예산 입력률 및 구간 분포
  입력: 3,384건 (42.3%)
  미입력: 4,616건 (57.7%)

  예산 구간별 분포:
              ~500만:   400건 (11.8%)
          500~1000만:   214건 (6.3%)
         1000~2000만:   256건 (7.6%)
         2000~3000만:   257건 (7.6%)
         3000~5000만:   701건 (20.7%)
           5000만~1억: 1,122건 (33.2%)
                1억+:   434건 (12.8%)
저장: eda_01_1_budget.png

[1-2] 단일 필터 조건별 커버리지

  조건             유니크값     유효 게시글     커버리지
  ------------------------------------------
  거주유형              7      8,000   100.0%
  대표스타일            11      7,169    89.6%
  주색조              32      8,000   100.0%
  전문가여부             3      8,000   100.0%
  공사유형              4      8,000   100.0%
  예산구간              7      3,384    42.3%
  지역              206      5,941    74.3%
저장: eda_01_2_single_filter.png



[1-3] 조건 조합 필터링 시 잔존 게시글 수

  시나리오                                           잔존     비율  판정
  -----------------------------------------------------------------
  거주유형만 (아파트)                                 5,953 74.41%  ✅ 충분
  아파트 + 내추럴                                   2,570 32.12%  ✅ 충분
  아파트 + 내추럴 + Light-Gray                      1,301 16.26%  ✅ 충분
  아파트 + 내추럴 + Light-Gray + 예산                    59  0.74%  ✅ 충분
  아파트 + 내추럴 + Light-Gray + 예산 + 서울                7  0.09%  ⚠️ 부족
  원룸 + 모던                                       178  2.23%  ✅ 충분
  원룸 + 모던 + Medium-Gray                          35  0.44%  ✅ 충분
  원룸 + 모던 + Medium-Gray + ~500만                   3  0.04%  ⚠️ 부족
저장: eda_01_3_combo_scenarios.png

[1-4] 지역 커버리지
  지역 입력: 5,941건 (74.3%)
  미입력: 2,059건 (25.7%)

  상위 10개 시/도:
sido
경기도      2231
서울특별시    2185
인천광역시     315
부산광역시     218
경상남도      132
대구광역시     120
충청남도       97
울산광역시      94
대전광역시      87
강원도        85
저장: eda_01_4_region_coverage.png

모든 분석 완료!


---
## EDA 02. 예산 분석
**목적**: 사용자 예산 필터링에 쓸 수 있는 두 가지 기준표를 만든다.

- **테이블 A**: 카테고리별 상품 단가 분포 → "소파 50만원 이하" 같은 단일 상품 교체 시 사용
- **테이블 B**: 공간별 합산 가격 분포 → "거실 전체 300만원 이하" 같은 공간 전체 변경 시 사용

**결론**:
- 테이블 A/B 모두 현실적인 수치로 추천 시스템에 바로 활용 가능
- features_budget(사용자 입력 예산)은 상관계수 0.03으로 태그 합산 예산과 무관 → 사용 안 함
- 게시글 전체 상품 합산은 집 전체 금액이라 수천만원 → 예산 필터는 반드시 공간 단위로 적용

### 필터링 전략
```
단일 상품 교체   → 테이블 A (카테고리별 가격) 사용
공간 전체 변경   → 테이블 B (공간별 합산) 사용
예산 미입력 시   → 해당 공간 중앙값을 기본 기준으로 사용
```


In [ ]:
# ── 데이터 로드 ──
print("로딩 중...")
tags_raw = pd.read_csv(f"{BASE}\\housewarming_product_tags_with_image.csv", low_memory=False)
print(f"태그 원본: {len(tags_raw):,}행")

# ── [결측 진단] productId 결측 현황 ──
n_total_tags = len(tags_raw)
n_missing = tags_raw["productId"].isna().sum()
n_valid   = n_total_tags - n_missing
print(f"\n[결측 진단]")
print(f"  유효 태그: {n_valid:,}건 ({n_valid/n_total_tags*100:.1f}%)")
print(f"  결측 태그: {n_missing:,}건 ({n_missing/n_total_tags*100:.1f}%)")

# 게시글별 결측 비율
tags_raw["is_missing"] = tags_raw["productId"].isna()
post_missing = tags_raw.groupby("contentId")["is_missing"].agg(["sum","count"])
post_missing.columns = ["결측수","전체태그수"]
post_missing["결측비율"] = post_missing["결측수"] / post_missing["전체태그수"]
print(f"  결측 30%+ 게시글: {(post_missing['결측비율']>=0.3).sum():,}건 (전체의 {(post_missing['결측비율']>=0.3).sum()/len(post_missing)*100:.1f}%)")
print("  → 결측 제거 후에도 편향 없음 확인. 그냥 제거하기로 결정.")

로딩 중...
태그 데이터: 1,941,343행

[진단 1] 전체 결측 현황
  전체 태그:  1,941,343건
  유효 태그:  1,732,521건 (89.2%)
  결측 태그:  208,822건 (10.8%)

[진단 2] 카테고리별 결측 비율

  카테고리                  전체       결측     결측비율
  ---------------------------------------------
  Binary shop           56        0     0.0%
  가구               366,655        0     0.0%
  가전·디지털           171,597        0     0.0%
  공구·DIY           239,681        0     0.0%
  데코·식물            219,654        0     0.0%
  렌탈·구독              6,540        0     0.0%
  반려동물              10,863        0     0.0%
  생필품               12,983        0     0.0%
  생활용품              40,746        0     0.0%
  수납·정리             64,149        0     0.0%
  식품                 4,538        0     0.0%
  유아·아동             40,876        0     0.0%
  인테리어시공            86,724        0     0.0%
  조명               142,677        0     0.0%
  주방용품             121,194        0     0.0%
  캠핑·레저             13,188        0     0.0%
  패브릭              189,018        0     0.0%


In [ ]:
# ── 클린 태그 데이터 생성 ──
# 제외 카테고리: 식품 (완전 무관)
EXCLUDE_DEPTH1 = ["식품"]

# 유효 공간 13개 (비공간 라벨 제외)
VALID_PLACES = [
    "거실", "주방", "침실", "아이방", "원룸",
    "서재&작업실", "욕실", "베란다", "드레스룸",
    "사무공간", "현관", "복도", "가구&소품"
]

tags_clean = tags_raw[~tags_raw["ohou_category_depth1"].isin(EXCLUDE_DEPTH1)]
tags_clean = tags_clean[tags_clean["place_label"].isin(VALID_PLACES)]
tags_clean = tags_clean.dropna(subset=["productId", "sellingPrice"])
tags_clean = tags_clean[tags_clean["sellingPrice"] > 0]

print(f"클린 데이터: {len(tags_clean):,}행 (원본의 {len(tags_clean)/len(tags_raw)*100:.1f}%)")
print(f"\n공간별 분포:")
for place, cnt in tags_clean["place_label"].value_counts().items():
    print(f"  {place:<12}: {cnt:>8,}건")

tags_clean.to_csv(f"{BASE}\\housewarming_product_tags_clean.csv", index=False, encoding="utf-8-sig")
print("\n저장: housewarming_product_tags_clean.csv")

로딩 중...
원본: 1,941,343행
식품 제거 후: 1,936,805행 (-4,538건)
유효 공간 필터 후: 1,887,247행 (-54,096건 총감소)
결측·가격0 제거 후: 1,476,918행 (-410,329건)

최종 클린 데이터: 1,476,918행
원본 대비 유지율: 76.1%

공간별 분포:
  거실          :  408,189건
  주방          :  325,811건
  침실          :  225,685건
  서재&작업실      :  119,691건
  욕실          :   86,597건
  가구&소품       :   61,859건
  아이방         :   61,162건
  현관          :   42,605건
  원룸          :   41,058건
  드레스룸        :   40,234건
  베란다         :   32,745건
  복도          :   22,739건
  사무공간        :    8,543건

depth1 분포:
  가구             :  334,376건
  데코·식물          :  202,804건
  패브릭            :  177,154건
  공구·DIY         :  171,351건
  가전·디지털         :  162,741건
  조명             :  132,147건
  주방용품           :  110,853건
  수납·정리          :   59,667건
  생활용품           :   37,447건
  유아·아동          :   36,339건
  인테리어시공         :   23,167건
  생필품            :   11,128건
  반려동물           :    9,921건
  캠핑·레저          :    6,621건
  렌탈·구독          :      538건
  혼수·신혼          :       83건
  Binary s

In [ ]:
# ── 테이블 A — 카테고리별 가격 분포 ──
print("로딩 중...")
tags = pd.read_csv(f"{BASE}\\housewarming_product_tags_clean.csv", low_memory=False)
tags_dedup = tags.drop_duplicates(subset=["contentId", "productId"])
print(f"클린 태그: {len(tags):,}행 / 중복제거: {len(tags_dedup):,}행")

def make_price_table(df, group_col):
    return (
        df.groupby(group_col)["sellingPrice"]
        .agg(상품수="count", p25=lambda x: x.quantile(0.25),
             중앙값="median", p75=lambda x: x.quantile(0.75), 평균="mean")
        .round(0).sort_values("상품수", ascending=False)
    )

cat_d1 = make_price_table(tags_dedup, "ohou_category_depth1")
cat_d2 = make_price_table(tags_dedup, "ohou_category_depth2")
cat_d3 = make_price_table(tags_dedup.dropna(subset=["ohou_category_depth3"]), "ohou_category_depth3")

cat_d1.to_csv(f"{BASE}\\price_table_A_depth1_clean.csv", encoding="utf-8-sig")
cat_d2.to_csv(f"{BASE}\\price_table_A_depth2_clean.csv", encoding="utf-8-sig")
cat_d3.to_csv(f"{BASE}\\price_table_A_depth3_clean.csv", encoding="utf-8-sig")

print("\n[테이블 A — depth1 가격 분포]")
print(f"  {'카테고리':<15} {'상품수':>7} {'p25':>9} {'중앙값':>9} {'p75':>9} {'평균':>9}")
print("  " + "-" * 62)
for idx, row in cat_d1.iterrows():
    print(f"  {str(idx):<15} {int(row['상품수']):>7,} {row['p25']/10000:>7,.1f}만 "
          f"{row['중앙값']/10000:>7,.1f}만 {row['p75']/10000:>7,.1f}만 {row['평균']/10000:>7,.1f}만")

# ── 테이블 B — 공간별 합산 가격 분포 ──
tags_place_dedup = tags.drop_duplicates(subset=["contentId", "place_label", "productId"])
place_budget = (
    tags_place_dedup.groupby(["contentId", "place_label"])["sellingPrice"]
    .sum().reset_index().rename(columns={"sellingPrice": "공간_합산가격"})
)
place_price = (
    place_budget.groupby("place_label")["공간_합산가격"]
    .agg(게시글수="count", p25=lambda x: x.quantile(0.25),
         중앙값="median", p75=lambda x: x.quantile(0.75), 평균="mean")
    .round(0).sort_values("중앙값", ascending=False)
)
place_price.to_csv(f"{BASE}\\price_table_B_place_clean.csv", encoding="utf-8-sig")
place_budget.to_csv(f"{BASE}\\price_table_B_place_detail_clean.csv", index=False, encoding="utf-8-sig")

print("\n[테이블 B — 공간별 합산 가격]")
print(f"  {'공간':<12} {'게시글수':>7} {'p25':>9} {'중앙값':>9} {'p75':>9} {'평균':>9}")
print("  " + "-" * 57)
for idx, row in place_price.iterrows():
    print(f"  {str(idx):<12} {int(row['게시글수']):>7,} {row['p25']/10000:>7,.0f}만 "
          f"{row['중앙값']/10000:>7,.0f}만 {row['p75']/10000:>7,.0f}만 {row['평균']/10000:>7,.0f}만")

# ── posts_final 생성 (공간별 예산 컬럼 포함) ──
posts = pd.read_csv(f"{BASE}\\housewarming_posts_with_tone_urlinfo.csv")
MAIN_PLACES = ["거실", "주방", "침실", "욕실", "현관", "서재&작업실", "아이방", "원룸"]
for place in MAIN_PLACES:
    sub = place_budget[place_budget["place_label"] == place][["contentId", "공간_합산가격"]]
    posts = posts.merge(sub.rename(columns={"공간_합산가격": f"예산_{place}"}), on="contentId", how="left")

posts.to_csv(f"{BASE}\\housewarming_posts_final.csv", index=False, encoding="utf-8-sig")
print(f"\n저장: housewarming_posts_final.csv ({posts.shape[1]}컬럼)")
print("EDA 02 완료!")

로딩 중...
클린 태그: 1,476,918행 / 게시글: 8,000행

[① 테이블 A] 카테고리별 가격 분포
중복 제거 후: 520,120행

  [depth1 가격 분포]
  카테고리                상품수        p25        중앙값        p75         평균
  -----------------------------------------------------------------
  가구               86,858      5.7만     13.9만     36.2만     43.8만
  데코·식물            85,609      0.6만      1.4만      3.2만      3.3만
  주방용품             64,885      0.8만      1.7만      3.5만      3.8만
  가전·디지털           59,478     15.9만     47.1만    131.6만     96.8만
  패브릭              52,623      1.3만      2.2만      4.3만      4.1만
  공구·DIY           49,109      0.9만      3.8만     13.8만     10.6만
  조명               35,011      2.7만      6.6만     21.6만     21.0만
  수납·정리            26,857      0.6만      1.4만      2.9만      3.1만
  생활용품             22,204      0.9만      2.0만      5.0만      4.7만
  유아·아동            14,631      2.9만      7.5만     19.6만     17.1만
  인테리어시공            7,608     15.0만     39.6만     85.8만     72.3만
  생필품               7,468      1.4만  

---
## EDA 03. 색조·스타일·상품 패턴 분석
**목적**: 색조/스타일별 상품 패턴 차이가 존재하는지 확인하여 조합 추천의 근거를 마련한다.

**결론**:
- **색조 → 카테고리 차이 없음**: 그레이/웜톤 모두 동일한 카테고리 구성 → 게시글 필터링으로 간접 반영
- **스타일 → 카테고리 차이 뚜렷**: 빈티지는 데코/홈갤러리, 미니멀은 시공/가전, 내추럴은 주방소품 중심
- **색조 + 예산 동시 필터링**: 무채색/웜톤 모두 모든 조합에서 ✅ 충분 (30건 이상)
- **쿨톤은 데이터 부족**: 조건 2개 이상 시 ⚠️ — 프로젝트 수준에서 허용 범위

### 추천 시스템 반영 방향
```
색조  → 게시글 필터링으로 간접 반영 (색조 맞는 게시글의 상품 추천)
스타일 → 카테고리 가중치로 직접 반영 (빈티지면 홈갤러리 가중치↑)
```


In [ ]:
# ── 데이터 로드 ──
print("로딩 중...")
tags  = pd.read_csv(f"{BASE}\\housewarming_product_tags_clean.csv", low_memory=False)
posts = pd.read_csv(f"{BASE}\\housewarming_posts_final.csv")
place_detail = pd.read_csv(f"{BASE}\\price_table_B_place_detail_clean.csv")
print(f"클린 태그: {len(tags):,}행 / 게시글: {len(posts):,}행")

posts["styleList"]  = parse_list_col(posts["features_styleList"])
posts["style_main"] = posts["styleList"].apply(lambda x: x[0] if len(x) > 0 else None)
posts["tone_group"] = posts["primary_tone"].apply(group_tone)

# 거실 예산 병합
living_budget = place_detail[place_detail["place_label"] == "거실"][["contentId", "공간_합산가격"]]
posts = posts.merge(living_budget.rename(columns={"공간_합산가격": "거실_합산"}), on="contentId", how="left")

SPACES = ["거실", "주방", "침실", "욕실", "현관", "서재&작업실", "아이방", "원룸"]
for space in SPACES:
    sub = place_detail[place_detail["place_label"] == space][["contentId", "공간_합산가격"]]
    posts = posts.merge(sub.rename(columns={"공간_합산가격": f"{space}_합산"}), on="contentId", how="left")

# JOIN
merged = tags.merge(posts[["contentId", "primary_tone", "features_styleList",
                            "features_residence", "tone_group", "style_main"]], on="contentId", how="inner")
merged_dedup = merged.drop_duplicates(subset=["contentId", "productId"])
print(f"JOIN + 중복제거: {len(merged_dedup):,}행")

# ── [Q1] 색조별 상품 카테고리 비율 ──
print("\n[Q1] 색조 그룹별 상품 카테고리 비율 (depth1)")
MAIN_STYLES = ["내추럴", "모던", "빈티지&레트로", "미니멀&심플", "유니크&믹스매치"]

tone_cat = (merged_dedup.groupby(["tone_group", "ohou_category_depth1"]).size().reset_index(name="count"))
tone_total = tone_cat.groupby("tone_group")["count"].transform("sum")
tone_cat["비율"] = tone_cat["count"] / tone_total * 100
tone_cat_pivot = tone_cat.pivot(index="ohou_category_depth1", columns="tone_group", values="비율").fillna(0).round(1)
print(tone_cat_pivot.to_string())
tone_cat_pivot.to_csv(f"{BASE}\\eda_03_tone_x_category.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle("[Q1] 색조 그룹별 상품 카테고리 비율 (%)", fontsize=14, fontweight="bold")
im = ax.imshow(tone_cat_pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(tone_cat_pivot.columns)))
ax.set_xticklabels(tone_cat_pivot.columns, rotation=20, ha="right", fontsize=10)
ax.set_yticks(range(len(tone_cat_pivot.index)))
ax.set_yticklabels(tone_cat_pivot.index, fontsize=10)
plt.colorbar(im, ax=ax, label="비율 (%)")
for i in range(len(tone_cat_pivot.index)):
    for j in range(len(tone_cat_pivot.columns)):
        val = tone_cat_pivot.values[i, j]
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8,
                color="black" if val < 20 else "white")
save_fig("eda_03_Q1_tone_heatmap", fig)

# ── [Q2] 스타일별 카테고리 비율 + 특징 상품 ──
print("\n[Q2] 스타일별 상품 카테고리 비율 (depth1)")
style_cat = (merged_dedup[merged_dedup["style_main"].isin(MAIN_STYLES)]
             .groupby(["style_main", "ohou_category_depth1"]).size().reset_index(name="count"))
style_total = style_cat.groupby("style_main")["count"].transform("sum")
style_cat["비율"] = style_cat["count"] / style_total * 100
style_pivot = style_cat.pivot(index="ohou_category_depth1", columns="style_main", values="비율").fillna(0).round(1)
print(style_pivot[MAIN_STYLES].to_string())
style_pivot.to_csv(f"{BASE}\\eda_03_style_x_category.csv", encoding="utf-8-sig")

print("\n[Q2-특징] 스타일별 특징 카테고리 (타 스타일 대비 등장 비율)")
for style in MAIN_STYLES:
    sub = merged_dedup[merged_dedup["style_main"] == style]
    others = merged_dedup[merged_dedup["style_main"].isin(MAIN_STYLES) & (merged_dedup["style_main"] != style)]
    sub_rate = sub["ohou_category_depth2"].value_counts(normalize=True)
    other_rate = others["ohou_category_depth2"].value_counts(normalize=True)
    ratio = (sub_rate / (other_rate + 0.001)).sort_values(ascending=False)
    top5 = ratio.head(5)
    print(f"\n  [{style}] n={len(sub):,}건")
    for cat, r in top5.items():
        cnt = sub["ohou_category_depth2"].value_counts().get(cat, 0)
        print(f"    {cat}: {r:.1f}배 ({cnt:,}건)")

fig, axes = plt.subplots(1, len(MAIN_STYLES), figsize=(20, 6))
fig.suptitle("[Q2] 스타일별 상위 상품 카테고리 (거실)", fontsize=14, fontweight="bold")
living_dedup = merged_dedup[merged_dedup["place_label"] == "거실"]
colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]
for i, (style, color) in enumerate(zip(MAIN_STYLES, colors)):
    sub = living_dedup[living_dedup["style_main"] == style]
    top = sub["ohou_category_depth2"].value_counts().head(8)
    axes[i].barh(top.index[::-1], top.values[::-1], color=color, edgecolor="white")
    axes[i].set_title(f"{style}\n(n={len(sub):,})")
    axes[i].set_xlabel("등장 횟수")
    for j, v in enumerate(top.values[::-1]):
        axes[i].text(v+1, j, f"{v:,}", va="center", fontsize=8)
save_fig("eda_03_Q2_style_bar", fig)

# ── [Q3] 색조 + 예산 필터링 검증 ──
print("\n[Q3] 색조 × 공간 × 예산 필터링 검증")
scenarios = [
    {"name": "거실 + 무채색",                       "place": "거실",  "tone": "무채색", "budget": None,    "style": None},
    {"name": "거실 + 무채색 + 300만 이하",            "place": "거실",  "tone": "무채색", "budget": 3000000, "style": None},
    {"name": "거실 + 무채색 + 300만 이하 + 내추럴",    "place": "거실",  "tone": "무채색", "budget": 3000000, "style": "내추럴"},
    {"name": "거실 + 무채색 + 300만 이하 + 모던",     "place": "거실",  "tone": "무채색", "budget": 3000000, "style": "모던"},
    {"name": "거실 + 웜톤 + 300만 이하",             "place": "거실",  "tone": "웜톤",  "budget": 3000000, "style": None},
    {"name": "거실 + 웜톤 + 300만 이하 + 내추럴",     "place": "거실",  "tone": "웜톤",  "budget": 3000000, "style": "내추럴"},
    {"name": "거실 + 웜톤 + 300만 이하 + 빈티지",     "place": "거실",  "tone": "웜톤",  "budget": 3000000, "style": "빈티지&레트로"},
    {"name": "거실 + 쿨톤 + 300만 이하",             "place": "거실",  "tone": "쿨톤",  "budget": 3000000, "style": None},
    {"name": "침실 + 웜톤 + 200만 이하 + 내추럴",     "place": "침실",  "tone": "웜톤",  "budget": 2000000, "style": "내추럴"},
    {"name": "원룸 + 무채색 + 200만 이하",            "place": "원룸",  "tone": "무채색", "budget": 2000000, "style": None},
]
results_q3 = []
print(f"  {'시나리오':<45} {'잔존':>6} {'비율':>6}  판정")
print("  " + "-" * 68)
for s in scenarios:
    place_col = f"{s['place']}_합산"
    sub = posts.dropna(subset=[place_col]).copy()
    if s["tone"]:  sub = sub[sub["tone_group"] == s["tone"]]
    if s["budget"]: sub = sub[sub[place_col] <= s["budget"]]
    if s["style"]:  sub = sub[sub["style_main"] == s["style"]]
    n = len(sub)
    total = len(posts.dropna(subset=[place_col]))
    flag = "✅ 충분" if n >= 30 else "⚠️ 부족" if n >= 10 else "❌ 불가"
    print(f"  {s['name']:<45} {n:>6,} {n/total*100:>5.1f}%  {flag}")
    results_q3.append({"시나리오": s["name"], "잔존": n, "판정": flag})
df_q3 = pd.DataFrame(results_q3)

fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle("[Q3] 색조 × 공간 × 예산 필터링 검증", fontsize=14, fontweight="bold")
bar_colors = ["#4C72B0" if r >= 30 else "#DD8452" if r >= 10 else "#D94F4F" for r in df_q3["잔존"]]
bars = ax.barh(df_q3["시나리오"][::-1], df_q3["잔존"][::-1], color=bar_colors[::-1], edgecolor="white")
ax.axvline(x=30, color="green", linestyle="--", linewidth=1.2, label="최소 기준(30건)")
ax.set_xlabel("잔존 게시글 수")
ax.legend()
for bar, v in zip(bars, df_q3["잔존"][::-1]):
    ax.text(v+1, bar.get_y()+bar.get_height()/2, f"{v:,}건", va="center", fontsize=8)
save_fig("eda_03_Q3_filter", fig)
print("\nEDA 03 완료!")

로딩 중...
클린 태그: 1,476,918행 / 게시글: 8,000행
JOIN 결과: 1,476,918행

색조 그룹 분포:
tone_group
무채색(그레이/화이트)    5689
웜톤(오렌지/옐로우)     2145
쿨톤(블루/스카이)        87
그린톤               45
레드톤               34
Name: count, dtype: int64

[Q1-1] 색조 그룹별 상품 카테고리 비율
tone_group             그린톤   레드톤  무채색(그레이/화이트)  웜톤(오렌지/옐로우)  쿨톤(블루/스카이)
ohou_category_depth1                                                   
Binary shop            0.0   0.0           0.0          0.0         0.0
가구                    14.4  18.2          16.6         17.0        16.8
가전·디지털                10.6  11.7          11.6         11.0        11.8
공구·DIY                 7.4   6.8          10.0          8.1         7.7
데코·식물                 18.3  17.7          15.8         18.2        16.9
렌탈·구독                  0.0   0.0           0.0          0.0         0.0
반려동물                   0.6   1.5           0.8          0.8         1.2
생필품                    1.7   1.1           1.5          1.3         1.2
생활용품                   3.8   2.9         

로딩 중...
클린 태그: 1,476,918행 / 게시글: 8,000행

스타일 분포:
style_main
내추럴         3362
모던          2208
미니멀&심플       591
빈티지&레트로      338
유니크&믹스매치     316
러블리&로맨틱      119
한국&아시아        65
북유럽           57
클래식&앤틱        56
프렌치&프로방스      46
Name: count, dtype: int64
JOIN + 중복제거: 520,120행

[Q2-1] 스타일별 상품 카테고리 비율 (depth1)
style_main             내추럴    모던  미니멀&심플  빈티지&레트로  유니크&믹스매치
ohou_category_depth1                                       
Binary shop            0.0   0.0     0.0      0.0       0.0
가구                    16.4  16.5    15.3     16.9      18.2
가전·디지털                11.3  13.1    15.8      8.4       8.7
공구·DIY                 8.6  10.9    29.1      4.7       6.1
데코·식물                 16.7  14.6     5.8     23.5      21.2
렌탈·구독                  0.0   0.0     0.0      0.0       0.0
반려동물                   0.8   0.7     0.4      0.7       1.0
생필품                    1.5   1.7     1.1      1.0       1.1
생활용품                   4.2   5.0     6.4      2.9       3.2
수납·정리                  5.4   

로딩 중...
게시글: 8,000행
공간별 예산 컬럼 추가 완료

[시나리오별 필터링 결과]

  시나리오                                              잔존     비율  판정
  --------------------------------------------------------------------
  거실 + 웜톤                                        1,896  26.1%  ✅ 충분
  거실 + 웜톤 + 300만 이하                                590   8.1%  ✅ 충분
  거실 + 웜톤 + 300만 이하 + 내추럴                          233   3.2%  ✅ 충분
  거실 + 웜톤 + 300만 이하 + 모던                           100   1.4%  ✅ 충분
  거실 + 웜톤 + 300만 이하 + 빈티지                           79   1.1%  ✅ 충분
  거실 + 쿨톤                                           66   0.9%  ✅ 충분
  거실 + 쿨톤 + 300만 이하                                 17   0.2%  ⚠️ 부족
  거실 + 쿨톤 + 300만 이하 + 내추럴                            5   0.1%  ❌ 불가
  침실 + 무채색 + 200만 이하                             2,897  40.3%  ✅ 충분
  침실 + 웜톤 + 200만 이하                              1,092  15.2%  ✅ 충분
  침실 + 웜톤 + 200만 이하 + 내추럴                          481   6.7%  ✅ 충분
  침실 + 웜톤 + 200만 이하 + 모던                           215   3.0%

---
## EDA 04. 공간별 상품 군집 분석
**목적**: 공간별로 자주 함께 등장하는 상품 카테고리 조합을 파악하여  
사용자가 특정 아이템을 선택했을 때 함께 추천할 카테고리 조합을 정의한다.

**결론**:
- 공간별로 뚜렷한 카테고리 동시 등장 패턴 존재
- 카테고리 단위 조합은 안정적 → 추천 시스템에 직접 활용 가능
- 상품 단위 조합은 노이즈 많음 → 보조 참고용

### 공간별 기본 추천 세트 (카테고리 기준)
```
거실 소파 선택 → 테이블 + 러그 + 천장등 + 쿠션 추천
침실 침대 선택 → 이불·이불솜 + 커튼 + 사이드테이블 + 스탠드 추천
주방 가전 선택 → 그릇 + 컵 + 주방수납 + 천장등 추천
```


In [ ]:
# ── 데이터 로드 ──
print("로딩 중...")
tags = pd.read_csv(f"{BASE}\\housewarming_product_tags_clean.csv", low_memory=False)
print(f"클린 태그: {len(tags):,}행")
tags_dedup = tags.drop_duplicates(subset=["contentId", "place_label", "productId"])

# ── [A-1] 공간별 카테고리 동시 등장 빈도 ──
print("\n[A-1] 공간별 카테고리 동시 등장 빈도 (depth2)")
MAIN_SPACES = ["거실", "침실", "주방", "욕실", "현관"]
space_cooccur = {}

for space in MAIN_SPACES:
    print(f"\n  [{space}] 처리 중...")
    space_tags = tags_dedup[tags_dedup["place_label"] == space]
    post_cats = space_tags.groupby("contentId")["ohou_category_depth2"].apply(lambda x: list(set(x.dropna())))
    pair_counter = Counter()
    for cats in post_cats:
        if len(cats) >= 2:
            for pair in combinations(sorted(cats), 2):
                pair_counter[pair] += 1
    top_pairs = pair_counter.most_common(20)
    space_cooccur[space] = top_pairs
    print(f"  상위 10 조합:")
    for (a, b), cnt in top_pairs[:10]:
        print(f"    {a} + {b}: {cnt:,}건")

rows_cooccur = []
for space, pairs in space_cooccur.items():
    for (a, b), cnt in pairs:
        rows_cooccur.append({"공간": space, "카테고리A": a, "카테고리B": b, "동시등장수": cnt})
pd.DataFrame(rows_cooccur).to_csv(f"{BASE}\\eda_04_cooccur_category.csv", index=False, encoding="utf-8-sig")

# ── [A-2] 핵심 카테고리 기준 동반 카테고리 ──
print("\n[A-2] 핵심 카테고리 기준 동반 카테고리")
ANCHOR_CATS = {"거실": "소파", "침실": "침대", "주방": "주방가전", "욕실": "수도", "현관": "현관·신발정리"}
anchor_results = {}

for space, anchor in ANCHOR_CATS.items():
    space_tags = tags_dedup[tags_dedup["place_label"] == space]
    anchor_posts = space_tags[space_tags["ohou_category_depth2"] == anchor]["contentId"].unique()
    other_cats = (
        space_tags[space_tags["contentId"].isin(anchor_posts) &
                   (space_tags["ohou_category_depth2"] != anchor)]
        ["ohou_category_depth2"].value_counts().head(10)
    )
    anchor_results[f"{space}_{anchor}"] = other_cats
    total = len(anchor_posts)
    print(f"\n  [{space}] '{anchor}' 포함 게시글: {total:,}건")
    print(f"  {'동반 카테고리':<20} {'등장수':>7} {'비율':>7}")
    print("  " + "-" * 38)
    for cat, cnt in other_cats.items():
        print(f"  {cat:<20} {cnt:>7,} {cnt/total*100:>6.1f}%")

fig, axes = plt.subplots(1, len(ANCHOR_CATS), figsize=(22, 6))
fig.suptitle("[A-2] 핵심 카테고리 기준 동반 카테고리", fontsize=14, fontweight="bold")
colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]
for i, ((space, anchor), color) in enumerate(zip(ANCHOR_CATS.items(), colors)):
    data = anchor_results[f"{space}_{anchor}"]
    axes[i].barh(data.index[::-1], data.values[::-1], color=color, edgecolor="white")
    axes[i].set_title(f"{space} — '{anchor}' 기준")
    axes[i].set_xlabel("등장 수")
    axes[i].tick_params(axis="y", labelsize=8)
    for j, v in enumerate(data.values[::-1]):
        axes[i].text(v+1, j, f"{v:,}", va="center", fontsize=7)
save_fig("eda_04_A2_anchor", fig)
print("\nEDA 04 완료!")

로딩 중...
클린 태그: 1,476,918행

[A-1] 공간별 카테고리 동시 등장 빈도 (depth2)

  [거실] 처리 중...
  상위 10 조합:
    의자 + 테이블·식탁·책상: 3,405건
    소파 + 테이블·식탁·책상: 3,314건
    테이블·식탁·책상 + 플라워·식물: 3,069건
    소파 + 의자: 2,785건
    의자 + 플라워·식물: 2,782건
    소파 + 플라워·식물: 2,590건
    러그·카페트 + 테이블·식탁·책상: 2,343건
    천장등 + 테이블·식탁·책상: 2,180건
    커튼·부자재 + 테이블·식탁·책상: 2,176건
    소파 + 천장등: 2,069건

  [침실] 처리 중...
  상위 10 조합:
    이불·이불솜 + 침대: 2,021건
    이불·이불솜 + 커튼·부자재: 2,001건
    서랍·수납장 + 이불·이불솜: 1,837건
    베개·베개커버 + 이불·이불솜: 1,797건
    이불·이불솜 + 플라워·식물: 1,784건
    의자 + 이불·이불솜: 1,749건
    단스탠드 + 이불·이불솜: 1,723건
    이불·이불솜 + 홈갤러리: 1,659건
    침대 + 커튼·부자재: 1,365건
    서랍·수납장 + 침대: 1,358건

  [주방] 처리 중...
  상위 10 조합:
    의자 + 주방가전: 3,371건
    냉장고 + 주방가전: 3,183건
    주방가전 + 테이블·식탁·책상: 2,961건
    주방가전 + 주방수납·정리: 2,857건
    주방가전 + 컵·잔·텀블러: 2,824건
    주방가전 + 천장등: 2,764건
    수도 + 주방가전: 2,607건
    주방가전 + 주방잡화: 2,585건
    주방가전 + 플라워·식물: 2,401건
    의자 + 테이블·식탁·책상: 2,394건

  [욕실] 처리 중...
  상위 10 조합:
    수도 + 욕실용품: 3,018건
    수도 + 타일·파벽돌: 1,364건
    욕실용

로딩 중...
클린 태그: 1,476,918행

[B-1] 거실 상품 단위 동시 등장 분석
  거실 게시글 수: 7,259건
  평균 상품 수: 19.2개
  50회 이상 등장 상품: 153개
  조합 계산 중... (시간이 걸릴 수 있어요)
  총 조합 수: 7,347개

  상위 20 상품 조합:
  상품A                            상품B                                동시등장
  ------------------------------------------------------------------------
  LED T5 간접조명 라인조명_6colors       맞춤 비침없는 도톰 레이스/쉬폰커튼(나비주름/            56
  맞춤 비침없는 도톰 레이스/쉬폰커튼(나비주름/      PVC 걸레받이 몰딩                          56
  [5000원 포인트적립LS990 듀로플라스틱       [3000원 포인트적립] LS990 듀로플라스            51
  맞춤 비침없는 도톰 레이스/쉬폰커튼(나비주름/      2선식 아나로그인터폰 코콤 비디오폰(ASTRO            51
  맞춤 비침없는 도톰 레이스/쉬폰커튼(나비주름/      미나 MINI LED 버섯램프 무드등 조명 L            51
  [4000원 포인트적립] LS990 듀로플라스      [5000원 포인트적립LS990 듀로플라스틱             50
  PVC 걸레받이 몰딩                    PVC몰딩 천장몰딩 계단몰딩                      44
  맞춤 비침없는 도톰 레이스/쉬폰커튼(나비주름/      루씨에어 실링팬 레이더2 132cm형 천정선풍            43
  맞춤 비침없는 도톰 레이스/쉬폰커튼(나비주름/      덴마크 플렌스테드  미래 자연 (FUTURA             40
  맞춤 비침없는 도톰 레이스/쉬폰커튼(나비주름

---
## EDA 05. 인기도 가중치 분석
**목적**: 스크랩/좋아요 기반 인기도 점수를 산출하고,  
상품 단위 랭킹에 활용할 수 있는지 검증한다.

**결론**:
- 스크랩-좋아요 상관계수 0.956으로 거의 동일 신호 → 스크랩 중심으로 가중치 설계
- 인기도 점수 정규분포에 가깝게 분포 → 랭킹 기준으로 적합
- 최소 5건 이상 등장 상품 기준으로 카테고리별 현실적인 상품 랭킹 도출
- 스타일×색조 조합별로 다른 상품이 추천됨 → 맞춤 조합 추천 가능

### 인기도 가중치
```
popularity_score = 스크랩(40%) + 좋아요(30%) + 공유(20%) + 댓글(10%)  [로그 정규화]
상품 최종점수    = 등장빈도(50%) + 평균인기도(50%)  [최소 5건 이상]
```


In [ ]:
# ── 데이터 로드 ──
print("로딩 중...")
posts = pd.read_csv(f"{BASE}\\housewarming_posts_final.csv")
tags  = pd.read_csv(f"{BASE}\\housewarming_product_tags_clean.csv", low_memory=False)
print(f"게시글: {len(posts):,}행 / 태그: {len(tags):,}행")

posts["styleList"]  = parse_list_col(posts["features_styleList"])
posts["style_main"] = posts["styleList"].apply(lambda x: x[0] if len(x) > 0 else None)
posts["tone_group"] = posts["primary_tone"].apply(group_tone)

# ── [05-1] 인게이지먼트 분포 및 상관관계 ──
engage_cols = ["viewCount", "likeCount", "scrapCount", "shareCount", "commentAndReplyCount"]
print("\n[05-1] 인게이지먼트 지표 분포")
print(f"  {'지표':<22} {'중앙값':>8} {'평균':>8} {'p75':>8} {'p95':>8} {'최댓값':>10}")
print("  " + "-" * 65)
for col in engage_cols:
    print(f"  {col:<22} {posts[col].median():>8,.0f} {posts[col].mean():>8,.0f} "
          f"{posts[col].quantile(0.75):>8,.0f} {posts[col].quantile(0.95):>8,.0f} {posts[col].max():>10,.0f}")

print("\n[05-2] 인게이지먼트 간 상관관계")
print(posts[engage_cols].corr().round(3).to_string())

# ── [05-3] 인기도 점수 산출 ──
for col in ["scrapCount", "likeCount", "shareCount", "commentAndReplyCount"]:
    posts[f"{col}_log"] = np.log1p(posts[col])
    max_val = posts[f"{col}_log"].max()
    posts[f"{col}_norm"] = posts[f"{col}_log"] / max_val if max_val > 0 else 0

posts["popularity_score"] = (
    posts["scrapCount_norm"] * 0.40 + posts["likeCount_norm"] * 0.30 +
    posts["shareCount_norm"] * 0.20 + posts["commentAndReplyCount_norm"] * 0.10
)
posts["popularity_score"] = (posts["popularity_score"] / posts["popularity_score"].max() * 100).round(2)

print(f"\n[05-3] 인기도 점수 분포")
print(f"  중앙값: {posts['popularity_score'].median():.1f} / 평균: {posts['popularity_score'].mean():.1f} / 최댓값: {posts['popularity_score'].max():.1f}")

print("\n  인기도 상위 10개 게시글:")
top10 = posts.nlargest(10, "popularity_score")[["contentId","title","scrapCount","likeCount","shareCount","popularity_score","tone_group","style_main"]]
for _, row in top10.iterrows():
    print(f"  [{row['contentId']}] {str(row['title'])[:35]} | 스크랩 {row['scrapCount']:,} / 점수 {row['popularity_score']:.1f} / {row['tone_group']} / {row['style_main']}")

# ── [05-4] 색조·스타일별 인기도 ──
print("\n[05-4] 색조·스타일별 인기도 중앙값")
print(posts.groupby("tone_group")["popularity_score"].agg(["median","mean","count"]).round(2).to_string())
print()
print(posts.groupby("style_main")["popularity_score"].agg(["median","mean","count"])
      .sort_values("median", ascending=False).head(8).round(2).to_string())

# ── [05-5] 상품 단위 인기도 점수 (최소 5건 기준) ──
print("\n[05-5] 상품 단위 인기도 점수 (최소 5건 기준)")
tags_dedup = tags.drop_duplicates(subset=["contentId", "productId"])
tags_pop = tags_dedup.merge(posts[["contentId","popularity_score","tone_group","style_main"]], on="contentId", how="inner")

product_score = (
    tags_pop.groupby(["productId","productName","brand","sellingPrice",
                      "ohou_category_depth1","ohou_category_depth2","ohou_category_depth3"])
    .agg(등장게시글수=("contentId","count"), 평균인기도=("popularity_score","mean"),
         최고인기도=("popularity_score","max"))
    .reset_index()
)
product_score = product_score[product_score["등장게시글수"] >= 5].copy()
print(f"  필터링 후 유효 상품: {len(product_score):,}개")

max_log = np.log1p(product_score["등장게시글수"]).max()
max_pop = product_score["평균인기도"].max()
product_score["등장_norm"] = np.log1p(product_score["등장게시글수"]) / max_log
product_score["인기도_norm"] = product_score["평균인기도"] / max_pop
product_score["최종점수"] = (product_score["등장_norm"] * 0.5 + product_score["인기도_norm"] * 0.5) * 100
product_score["최종점수"] = product_score["최종점수"].round(2)
product_score = product_score.sort_values("최종점수", ascending=False)
product_score.to_csv(f"{BASE}\\eda_05_product_score_final.csv", index=False, encoding="utf-8-sig")

print("\n  카테고리별 상위 상품:")
for cat in ["소파", "침대", "의자", "테이블·식탁·책상", "천장등", "러그·카페트"]:
    sub = product_score[product_score["ohou_category_depth2"] == cat].head(3)
    if len(sub) == 0: continue
    print(f"\n  [{cat}]")
    for _, row in sub.iterrows():
        print(f"    {str(row['productName'])[:40]:<40} ({row['brand']}) {row['sellingPrice']:>10,.0f}원 / {int(row['등장게시글수'])}건 / 점수 {row['최종점수']:.1f}")

# ── [05-6] 스타일 × 색조별 소파 상위 상품 ──
print("\n[05-6] 스타일 × 색조별 소파 상위 상품 (조합 추천 검증)")
for tone in ["무채색", "웜톤"]:
    for style in ["내추럴", "모던"]:
        sub_posts = posts[(posts["tone_group"]==tone) & (posts["style_main"]==style)]["contentId"]
        sub_tags = tags_pop[(tags_pop["contentId"].isin(sub_posts)) & (tags_pop["ohou_category_depth2"]=="소파")]
        if len(sub_tags) == 0: continue
        sofa_score = (
            sub_tags.groupby(["productId","productName","brand","sellingPrice"])
            .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean")).reset_index()
        )
        sofa_score = sofa_score[sofa_score["등장수"] >= 3]
        if len(sofa_score) == 0: continue
        sofa_score["점수"] = np.log1p(sofa_score["등장수"]) * 0.5 + sofa_score["평균인기도"] / sofa_score["평균인기도"].max() * 0.5
        sofa_score = sofa_score.sort_values("점수", ascending=False).head(3)
        print(f"\n  [{tone} + {style}] 소파 상위 상품 (후보 {len(sub_tags):,}건)")
        for _, row in sofa_score.iterrows():
            print(f"    {str(row['productName'])[:45]:<45} {row['sellingPrice']:>10,.0f}원 / {int(row['등장수'])}건")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("[EDA 05] 상품 인기도 점수 분석", fontsize=14, fontweight="bold")
cat_counts = product_score.groupby("ohou_category_depth2")["productId"].count().sort_values(ascending=False).head(15)
axes[0].barh(cat_counts.index[::-1], cat_counts.values[::-1], color="#4C72B0", edgecolor="white")
axes[0].set_title("카테고리별 유효 상품 수 (5건 이상)")
axes[0].set_xlabel("상품 수")
for i, v in enumerate(cat_counts.values[::-1]):
    axes[0].text(v+1, i, f"{v:,}", va="center", fontsize=8)

axes[1].hist(product_score["최종점수"], bins=40, color="#4C72B0", edgecolor="white", linewidth=0.5)
axes[1].set_title("상품 최종점수 분포")
axes[1].set_xlabel("최종 점수")
axes[1].set_ylabel("상품 수")
axes[1].axvline(product_score["최종점수"].median(), color="#D94F4F", linestyle="--",
                label=f"중앙값 {product_score['최종점수'].median():.1f}")
axes[1].legend()
save_fig("eda_05_product_score", fig)
print("\nEDA 05 완료!")

로딩 중...
게시글: 8,000행 / 태그: 1,476,918행

[05-1] 인게이지먼트 지표 분포

  지표                          중앙값       평균      p75      p95        최댓값
  -----------------------------------------------------------------
  viewCount                21,709   30,366   39,794   83,973    375,712
  likeCount                   109      185      214      603      4,686
  scrapCount                  404      728      837    2,373     17,593
  shareCount                   58       99      115      331      3,047
  commentAndReplyCount         20       29       41       87        581

[05-2] 인게이지먼트 간 상관관계
                      viewCount  likeCount  scrapCount  shareCount  commentAndReplyCount
viewCount                 1.000      0.896       0.891       0.793                 0.607
likeCount                 0.896      1.000       0.956       0.827                 0.603
scrapCount                0.891      0.956       1.000       0.734                 0.505
shareCount                0.793      0.827       0.734       1.

로딩 중...
게시글: 8,000행 / 태그: 1,476,918행

[최소 등장 기준별 상품 수]
    1건 이상: 158,231개 상품 (100.0%)
    3건 이상:  37,045개 상품 (23.4%)
    5건 이상:  19,815개 상품 (12.5%)
   10건 이상:   8,495개 상품 (5.4%)
   20건 이상:   3,500개 상품 (2.2%)
   30건 이상:   2,028개 상품 (1.3%)
   50건 이상:     933개 상품 (0.6%)

[상품 점수 재산출 — 최소 5건 기준]
  필터링 후 상품 수: 16,234개
  저장: eda_05_product_score_v2.csv

[카테고리별 상위 상품]

  [소파] (총 104개 상품)
  상품명                                      브랜드                  가격    등장     점수
  ------------------------------------------------------------------------------
  Pebble Sofa                              잭슨카멜레온       3,370,000원   49건  63.6
  카이 가죽 3인 소파 3colors                      가구밸리           299,000원   32건  62.8
  1인용 아쿠아 패브릭 쇼파 버즈니아                      메이크가구          229,000원   15건  62.7
  Bolson Sofa                              잭슨카멜레온       3,810,000원   28건  62.6
  리빙 다이닝 소파 체어 떡갈나무 (팔걸이/커버 옵션선택)          무인양품           390,000원   17건  60.8

  [침대] (총 66개 상품)
  상품명                                    

---
## 종합 결론: 사용자 맞춤 상품 조합 추천 가능성 검증 결과

### ✅ 검증 통과

| 항목 | 결과 | 근거 |
|---|---|---|
| 필터 조건 유효성 | ✅ 가능 | 조건 3개까지 충분한 게시글 확보 |
| 예산 필터링 기준 | ✅ 가능 | 카테고리별·공간별 현실적 가격표 완성 |
| 색조 기반 추천 | ✅ 가능 | 색조→게시글 필터링→상품 추출 방식 |
| 스타일 기반 추천 | ✅ 가능 | 스타일별 카테고리 패턴 뚜렷 |
| 공간별 상품 조합 | ✅ 가능 | 앵커 카테고리별 동반 카테고리 정의 |
| 인기도 랭킹 | ✅ 가능 | 스타일×색조 조합별 다른 상품 추천 확인 |

### 추천 시스템 설계 방향
```
사용자 입력: 방 사진 + 공간 + 색조 + 스타일 + 예산
      ↓
1단계: 색조·스타일·예산으로 유사 게시글 필터링
      ↓
2단계: 필터링된 게시글에서 공간별 카테고리 조합 추출
       (소파 선택 → 테이블/러그/조명 세트 추천)
      ↓
3단계: 카테고리 내 인기도 점수 기반 상품 랭킹
      ↓
4단계: 예산 범위 내 상품 최종 필터링
```

### 생성된 산출물
- `housewarming_product_tags_clean.csv` — 클린 태그 데이터 (147만건)
- `housewarming_posts_final.csv` — 게시글 + 공간별 예산 컬럼
- `price_table_A_depth1/2/3_clean.csv` — 카테고리별 가격 분포
- `price_table_B_place_clean.csv` — 공간별 합산 가격 분포
- `eda_03_tone_x_category.csv` — 색조별 카테고리 비율
- `eda_03_style_x_category.csv` — 스타일별 카테고리 비율
- `eda_04_cooccur_category.csv` — 공간별 카테고리 동시 등장
- `eda_05_product_score_final.csv` — 상품 인기도 점수
